# 04 Tầng 3 — LLM Inference & So sánh

**Mục tiêu:**
- Nhận context từ Tầng 2 (RAG) và gọi 3 LLM đề xuất itinerary
- **GPT-4o-mini** (OpenAI) · **Claude Haiku 3.5** (Anthropic) · **Gemini 2.0 Flash** (Google)
- Parse kết quả về dạng danh sách POI chuẩn hóa
- Tính **Recall, Precision, F-score, AE, UE** cho từng mô hình
- So sánh với **Actual** (sequence thực tế) và **+Tour gốc** (kết quả GraphFrames)
- Xuất bảng kết quả tổng hợp

**API keys cần có:**
```bash
export OPENAI_API_KEY="sk-..."
export ANTHROPIC_API_KEY="sk-ant-..."
export GOOGLE_API_KEY="AIza..."
```

**Thư viện cần cài:**
```bash
pip install openai anthropic google-generativeai pandas
```


## 1. Cài đặt thư viện & cấu hình API keys

In [ ]:
import subprocess, sys

pkgs = ["openai", "anthropic", "google-generativeai", "pandas", "numpy"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

print("✓ Đã cài xong.")

In [ ]:
import os

# ── Cài API keys ───────────────────────────────────────────────────────────
# Cách 1: đặt biến môi trường trước khi chạy notebook
# Cách 2: uncomment và điền trực tiếp (KHÔNG commit lên git)

# os.environ["OPENAI_API_KEY"]    = "sk-..."
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["GOOGLE_API_KEY"]    = "AIza..."

OPENAI_KEY    = os.getenv("OPENAI_API_KEY",    "")
ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY", "")
GOOGLE_KEY    = os.getenv("GOOGLE_API_KEY",    "")

print("OpenAI   :", "✓" if OPENAI_KEY    else "✗ Chưa đặt OPENAI_API_KEY")
print("Anthropic:", "✓" if ANTHROPIC_KEY else "✗ Chưa đặt ANTHROPIC_API_KEY")
print("Google   :", "✓" if GOOGLE_KEY    else "✗ Chưa đặt GOOGLE_API_KEY")

## 2. Load context từ Tầng 2

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

current_path = Path.cwd()
project_root = current_path.parent if current_path.name.lower() == "notebooks" else current_path

rag_path    = project_root / "Datasets" / "RAG"
raw_path    = project_root / "Datasets" / "Raw"
result_path = project_root / "Datasets" / "Results"
result_path.mkdir(parents=True, exist_ok=True)

with open(rag_path / "llm_contexts.json", encoding="utf-8") as f:
    all_contexts = json.load(f)

print(f"Đã load {len(all_contexts)} contexts")
print(f"Cities : {set(c['city'] for c in all_contexts)}")

# Preview
ctx0 = all_contexts[0]
print(f"\nVí dụ context:")
print(f"  user_id : {ctx0['user_id']}")
print(f"  city    : {ctx0['city']}")
print(f"  budget  : {ctx0['budget_minutes']} phút")
print(f"  POIs retrieved: {len(ctx0['retrieved_pois'])}")

## 3. Prompt Template chuẩn cho tất cả mô hình

In [ ]:
SYSTEM_PROMPT = """You are a smart tourism itinerary planner. 
Given a tourist's profile and a list of recommended Points of Interest (POIs), 
your task is to create the best possible one-day itinerary.

Rules:
1. Select 3-6 POIs that fit within the time budget.
2. Order them logically (by location proximity or thematic flow).
3. Each selected POI must be from the PROVIDED LIST only.
4. Account for average visit duration of each POI.
5. Return ONLY a JSON object, no explanation, no markdown.

Output format (strict JSON):
{"itinerary": ["POI Name 1", "POI Name 2", "POI Name 3"], "total_time_min": 240, "reason": "brief reason"}"""


def build_user_prompt(ctx: dict) -> str:
    """Tạo user message từ context."""
    visited = ctx["user_profile"]["visited_pois"]
    top_cats = ctx["user_profile"]["top_categories"]

    cat_lines = "\n".join(
        f"  - {c['poiTheme']}: interest score {c['int_u_c']:.2f}"
        for c in top_cats
    ) if top_cats else "  - No strong preferences detected"

    poi_lines = "\n".join(
        f"  {i+1}. {p['name']} [{p['category']}] "
        f"~{p['avg_visit_min']:.0f}min, relevance={p['similarity']:.3f}. "
        f"{p['description']}"
        for i, p in enumerate(ctx["retrieved_pois"])
    )

    visited_str = ", ".join(visited) if visited else "None yet"

    return f"""Tourist profile:
- City: {ctx['city']}
- Time budget: {ctx['budget_minutes']} minutes
- Previously visited: {visited_str}
- Interest categories:
{cat_lines}

Available POIs to choose from:
{poi_lines}

Create the optimal itinerary. Return ONLY valid JSON."""


# Test prompt
print("=== SYSTEM PROMPT ===")
print(SYSTEM_PROMPT)
print("\n=== USER PROMPT (sample) ===")
print(build_user_prompt(ctx0)[:800], "...")

## 4. Wrappers gọi từng LLM

In [ ]:
import time, re

# ── Helpers ────────────────────────────────────────────────────────────────
def parse_llm_response(raw: str) -> dict:
    """Parse JSON từ response, xử lý markdown code blocks nếu có."""
    # Loại bỏ markdown code fences nếu có
    clean = re.sub(r"```(?:json)?\n?", "", raw).strip()
    # Lấy đoạn JSON đầu tiên trong response
    match = re.search(r"\{.*\}", clean, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return {"itinerary": [], "total_time_min": 0, "reason": "parse_error", "raw": raw[:500]}


# ── GPT-4o-mini ────────────────────────────────────────────────────────────
def call_gpt4o_mini(ctx: dict, max_tokens: int = 512) -> dict:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_KEY)

    t0 = time.time()
    resp = client.chat.completions.create(
        model       = "gpt-4o-mini",
        max_tokens  = max_tokens,
        temperature = 0.2,
        messages    = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_user_prompt(ctx)},
        ],
    )
    latency = time.time() - t0
    raw_text = resp.choices[0].message.content
    result = parse_llm_response(raw_text)
    result.update({
        "model":           "gpt-4o-mini",
        "latency_sec":     round(latency, 2),
        "prompt_tokens":   resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
    })
    return result


# ── Claude Haiku 3.5 ──────────────────────────────────────────────────────
def call_claude_haiku(ctx: dict, max_tokens: int = 512) -> dict:
    import anthropic
    client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

    t0 = time.time()
    resp = client.messages.create(
        model      = "claude-haiku-4-5-20251001",
        max_tokens = max_tokens,
        system     = SYSTEM_PROMPT,
        messages   = [{"role": "user", "content": build_user_prompt(ctx)}],
    )
    latency = time.time() - t0
    raw_text = resp.content[0].text
    result = parse_llm_response(raw_text)
    result.update({
        "model":             "claude-haiku-4-5",
        "latency_sec":       round(latency, 2),
        "prompt_tokens":     resp.usage.input_tokens,
        "completion_tokens": resp.usage.output_tokens,
    })
    return result


# ── Gemini 2.0 Flash ──────────────────────────────────────────────────────
def call_gemini_flash(ctx: dict, max_tokens: int = 512) -> dict:
    import google.generativeai as genai
    genai.configure(api_key=GOOGLE_KEY)
    model = genai.GenerativeModel(
        model_name="gemini-2.0-flash",
        system_instruction=SYSTEM_PROMPT,
    )

    t0 = time.time()
    resp = model.generate_content(
        build_user_prompt(ctx),
        generation_config=genai.types.GenerationConfig(
            max_output_tokens=max_tokens,
            temperature=0.2,
        ),
    )
    latency = time.time() - t0
    raw_text = resp.text
    result = parse_llm_response(raw_text)
    result.update({
        "model":             "gemini-2.0-flash",
        "latency_sec":       round(latency, 2),
        "prompt_tokens":     resp.usage_metadata.prompt_token_count   if resp.usage_metadata else 0,
        "completion_tokens": resp.usage_metadata.candidates_token_count if resp.usage_metadata else 0,
    })
    return result


# ── Dispatch ──────────────────────────────────────────────────────────────
MODEL_CALLERS = {
    "gpt-4o-mini":      call_gpt4o_mini,
    "claude-haiku-4-5": call_claude_haiku,
    "gemini-2.0-flash": call_gemini_flash,
}

def call_llm_safe(model_name: str, ctx: dict) -> dict:
    """Gọi LLM với error handling và retry."""
    for attempt in range(3):
        try:
            return MODEL_CALLERS[model_name](ctx)
        except Exception as e:
            print(f"    [{model_name}] Lỗi lần {attempt+1}: {e}")
            if attempt < 2:
                time.sleep(2 ** attempt)  # exponential backoff
    return {
        "model":     model_name,
        "itinerary": [],
        "reason":    "error",
        "latency_sec": 0,
        "prompt_tokens": 0,
        "completion_tokens": 0,
    }


print("✓ Các hàm gọi LLM đã sẵn sàng.")

## 5. Metrics đánh giá (theo bài báo)

In [ ]:
def normalize_poi_name(name: str) -> str:
    """Chuẩn hóa tên POI để so sánh."""
    return name.strip().lower().replace(" ", "_").replace("-", "_")


def compute_itinerary_metrics(
    predicted: list,    # danh sách POI tên (string) do LLM/model đề xuất
    actual: list,       # danh sách POI tên thực tế (ground truth)
) -> dict:
    """Tính Recall, Precision, F-score (Section 7.2 bài báo)."""
    pred_set  = set(normalize_poi_name(p) for p in predicted)
    actual_set = set(normalize_poi_name(a) for a in actual)

    if not actual_set or not pred_set:
        return {"recall": 0.0, "precision": 0.0, "f_score": 0.0}

    tp = len(pred_set & actual_set)
    recall    = tp / len(actual_set)
    precision = tp / len(pred_set)
    f_score   = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0.0
    )
    return {
        "recall":    round(recall,    4),
        "precision": round(precision, 4),
        "f_score":   round(f_score,   4),
    }


def compute_ae_ue(
    itinerary: list,
    profit_per_poi: dict,   # {poi_name_normalized: profit_score}
    max_profit: float = 1.0,
) -> dict:
    """
    AE (Allocation Efficiency) và UE (User Experience) — Eq. 22, 23 bài báo.
    Phiên bản đơn giản hóa cho LLM (không có MEC bandwidth thực):
    - AE ~ trung bình profit normalize của các POI được chọn
    - UE = AE * norm_profit_sum
    """
    if not itinerary:
        return {"ae": 0.0, "ue": 0.0}

    profits = [
        profit_per_poi.get(normalize_poi_name(p), 0.0)
        for p in itinerary
    ]
    ae  = sum(profits) / (len(profits) * max_profit) if max_profit > 0 else 0.0
    ue  = ae * (sum(profits) / max(max_profit * len(profits), 1))
    return {"ae": round(ae, 4), "ue": round(ue, 4)}


print("✓ Các hàm metrics đã sẵn sàng.")

## 6. Load Actual sequence và Profit map

In [ ]:
# Load ground truth: sequence thực tế của mỗi user
visit_list = []
for city_dir in sorted(raw_path.iterdir()):
    if not city_dir.is_dir():
        continue
    f = city_dir / "touristsVisits.csv"
    if f.exists():
        df = pd.read_csv(f)
        df["city"] = city_dir.name
        visit_list.append(df)
visit_pdf = pd.concat(visit_list, ignore_index=True)

poi_list = []
for city_dir in sorted(raw_path.iterdir()):
    if not city_dir.is_dir():
        continue
    f = city_dir / "POIs.csv"
    if f.exists():
        df = pd.read_csv(f)
        df["city"] = city_dir.name
        poi_list.append(df)
poi_pdf = pd.concat(poi_list, ignore_index=True)

# Profit đơn giản: dùng visit count normalize làm proxy
pop_count = (
    visit_pdf
    .groupby(["city", "poiID"])["userID"].count()
    .reset_index(name="pop")
)
profit_proxy = pop_count.merge(poi_pdf[["city","poiID","poiName"]], on=["city","poiID"])

# Normalize theo city
city_max = profit_proxy.groupby("city")["pop"].max().rename("pop_max")
profit_proxy = profit_proxy.join(city_max, on="city")
profit_proxy["profit_norm"] = profit_proxy["pop"] / profit_proxy["pop_max"]

# Dict: {(city, poi_name_normalized): profit_norm}
profit_map = {
    (row["city"], normalize_poi_name(row["poiName"])): row["profit_norm"]
    for _, row in profit_proxy.iterrows()
}

print(f"Profit map: {len(profit_map):,} entries")
print(f"Visits    : {len(visit_pdf):,} rows")

## 7. Chạy toàn bộ inference

In [ ]:
# ── Xác định mô hình nào khả dụng ─────────────────────────────────────────
MODELS_TO_RUN = []
if OPENAI_KEY:    MODELS_TO_RUN.append("gpt-4o-mini")
if ANTHROPIC_KEY: MODELS_TO_RUN.append("claude-haiku-4-5")
if GOOGLE_KEY:    MODELS_TO_RUN.append("gemini-2.0-flash")

if not MODELS_TO_RUN:
    print("⚠ Chưa đặt API key nào. Chạy ở chế độ MOCK để test pipeline.")
    MOCK_MODE = True
else:
    MOCK_MODE = False
    print(f"Sẽ gọi: {MODELS_TO_RUN}")


def mock_llm_response(model: str, ctx: dict) -> dict:
    """Mock response để test pipeline khi chưa có API key."""
    import random
    pois = ctx["retrieved_pois"]
    chosen = random.sample(pois, min(4, len(pois)))
    return {
        "model":             model,
        "itinerary":         [p["name"] for p in chosen],
        "total_time_min":    sum(p["avg_visit_min"] for p in chosen),
        "reason":            "mock response",
        "latency_sec":       0.1,
        "prompt_tokens":     100,
        "completion_tokens": 50,
    }

In [ ]:
all_results = []
DELAY_BETWEEN_CALLS = 1.5  # giây, tránh rate limit

# Chọn models
active_models = MODELS_TO_RUN if not MOCK_MODE else ["gpt-4o-mini", "claude-haiku-4-5", "gemini-2.0-flash"]

for ctx_idx, ctx in enumerate(all_contexts):
    user_id = ctx["user_id"]
    city    = ctx["city"]

    print(f"\n[{ctx_idx+1}/{len(all_contexts)}] User={user_id} | City={city}")

    # ── Actual sequence từ visit data ─────────────────────────────────────
    actual_seq = (
        visit_pdf[(visit_pdf["userID"] == user_id) & (visit_pdf["city"] == city)]
        .merge(poi_pdf[["city","poiID","poiName"]], on=["city","poiID"])
        .sort_values("unixTimestamp")
        ["poiName"].drop_duplicates().tolist()
    )

    base_row = {
        "user_id":    user_id,
        "city":       city,
        "budget_min": ctx["budget_minutes"],
        "actual_seq": actual_seq,
    }

    # ── Gọi từng LLM ──────────────────────────────────────────────────────
    for model in active_models:
        print(f"  → {model}...", end=" ")

        if MOCK_MODE:
            llm_out = mock_llm_response(model, ctx)
        else:
            llm_out = call_llm_safe(model, ctx)
            time.sleep(DELAY_BETWEEN_CALLS)

        # Tính metrics
        itinerary = llm_out.get("itinerary", [])
        city_profit_map = {
            k[1]: v for k, v in profit_map.items() if k[0] == city
        }

        trad_metrics  = compute_itinerary_metrics(itinerary, actual_seq)
        alloc_metrics = compute_ae_ue(
            itinerary,
            profit_per_poi=city_profit_map,
            max_profit=1.0,
        )

        row = {
            **base_row,
            "model":              llm_out["model"],
            "predicted_itinerary": itinerary,
            "predicted_time_min": llm_out.get("total_time_min", 0),
            "reason":             llm_out.get("reason", ""),
            "latency_sec":        llm_out.get("latency_sec", 0),
            "prompt_tokens":      llm_out.get("prompt_tokens", 0),
            "completion_tokens":  llm_out.get("completion_tokens", 0),
            **trad_metrics,
            **alloc_metrics,
        }
        all_results.append(row)

        print(f"recall={trad_metrics['recall']:.2f} "
              f"prec={trad_metrics['precision']:.2f} "
              f"f1={trad_metrics['f_score']:.2f} "
              f"ae={alloc_metrics['ae']:.2f} "
              f"latency={llm_out.get('latency_sec',0):.1f}s")

print(f"\n✓ Đã chạy {len(all_results)} rows.")

## 8. Lưu kết quả và xuất bảng so sánh

In [ ]:
results_df = pd.DataFrame(all_results)

# Lưu chi tiết
results_df.to_csv(result_path / "llm_inference_results.csv", index=False)
results_df.to_json(result_path / "llm_inference_results.json", orient="records", indent=2)

print("✓ Đã lưu llm_inference_results.csv và .json")
print(f"  Shape: {results_df.shape}")
results_df.head()

In [ ]:
# ── Bảng tổng hợp so sánh các mô hình ────────────────────────────────────
metrics_cols = ["recall", "precision", "f_score", "ae", "ue",
                "latency_sec", "prompt_tokens", "completion_tokens"]

summary = (
    results_df
    .groupby("model")[metrics_cols]
    .agg(["mean", "std"])
    .round(4)
)

print("=" * 80)
print("BẢNG TỔNG HỢP SO SÁNH CÁC MÔ HÌNH")
print("=" * 80)
print(summary.to_string())

In [ ]:
# ── Bảng gọn theo từng city ───────────────────────────────────────────────
city_model_summary = (
    results_df
    .groupby(["city", "model"])[["recall", "precision", "f_score", "ae", "ue"]]
    .mean()
    .round(4)
    .reset_index()
)

print("\nKết quả theo thành phố:")
print(city_model_summary.to_string(index=False))

city_model_summary.to_csv(result_path / "summary_by_city.csv", index=False)
print("\n✓ Đã lưu summary_by_city.csv")

In [ ]:
# ── Bảng chi phí token & latency ──────────────────────────────────────────
cost_summary = (
    results_df
    .groupby("model")[["latency_sec", "prompt_tokens", "completion_tokens"]]
    .mean()
    .round(2)
)
cost_summary["total_tokens"] = (
    cost_summary["prompt_tokens"] + cost_summary["completion_tokens"]
)

# Ước tính chi phí (giá tháng 5/2026, USD per 1M tokens)
PRICE_PER_1M = {
    "gpt-4o-mini":      {"input": 0.15,  "output": 0.60},
    "claude-haiku-4-5": {"input": 0.80,  "output": 4.00},
    "gemini-2.0-flash": {"input": 0.075, "output": 0.30},
}

for model, row in cost_summary.iterrows():
    price = PRICE_PER_1M.get(model, {"input": 0, "output": 0})
    cost = (
        row["prompt_tokens"]     / 1e6 * price["input"] +
        row["completion_tokens"] / 1e6 * price["output"]
    )
    cost_summary.loc[model, "cost_usd_per_call"] = round(cost, 6)

print("\nChi phí và latency trung bình:")
print(cost_summary.to_string())

cost_summary.to_csv(result_path / "cost_summary.csv")
print("\n✓ Tầng 3 hoàn thành. Kết quả lưu tại:", result_path)

## 9. Kết luận nhanh

In [ ]:
best_fscore = results_df.groupby("model")["f_score"].mean()
best_ae     = results_df.groupby("model")["ae"].mean()
best_ue     = results_df.groupby("model")["ue"].mean()

print("=" * 50)
print("RANKING CÁC MÔ HÌNH")
print("=" * 50)
print("\nTheo F-score (itinerary accuracy):")
print(best_fscore.sort_values(ascending=False).to_string())
print("\nTheo AE (allocation efficiency):")
print(best_ae.sort_values(ascending=False).to_string())
print("\nTheo UE (user experience):")
print(best_ue.sort_values(ascending=False).to_string())
print("\n→ Tầng 4 (Đánh giá tổng hợp) sẽ vẽ biểu đồ và so sánh với +Tour gốc.")